<a href="https://colab.research.google.com/github/shubham-bari/Language-Models/blob/main/Optimizers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

class Adam(nn.Module):

  def __init__(self, lr=0.001, decay=0.1, eps=1e-8, beta1=0.9, beta2=0.999):
    super().__init__()
    self.beta1 = beta1
    self.beta2 = beta2
    self.decay = decay
    self.lr = lr
    self.iterations=0
    self.eps = eps

  #call once before update starts
  def pre_update_params(self):

    if self.decay:
      self.curr_lr = self.lr * (1/(1+self.decay*self.iterations))

  def update_params(self, layer):

    if not hasattr(layer, 'weight_cache'):

      layer.weight_momentums = torch.zeros_like(layer.weights)
      layer.bias_momentums = torch.zeros_like(layer.bias)

      layer.weight_cache = torch.zeros_like(layer.weights)
      layer.bias_cache = torch.zeros_like(layer.weights)

    layer.weight_momentums = self.beta1 * layer.weight_momentums + (1-self.beta1) * layer.dweights   #dweights is loss gradient for that layer
    layer.bias_momentums = self.beta1 * layer.bias_momentums + (1-self.beta1) * layer.dbias

    layer.weight_cache = self.beta2*layer.weight_cache + (1-self.beta2) * layer.dweights**2
    layer.bias_cache = self.beta2*layer.bias_cache + (1-self.beta2) * layer.dbias**2

    corrected_weight_momentums = layer.weight_momentums/(1-self.beta1**self.iterations)
    corrected_bias_momentums = layer.bias_momentums/(1-self.beta1**self.iterations)

    corrected_weight_cache = layer.weight_cache/(1-self.beta2**self.iterations)
    corrected_bias_cache = layer.bias_cache/(1-self.beta2**self.iterations)

    layer.weights = layer.weights - self.curr_lr * (corrected_weight_momentums/(torch.sqrt(corrected_weight_cache+eps)))
    layer.bias = layer.bias - self.curr_lr * (corrected_bias_momentums/(torch.sqrt(corrected_bias_cache+eps)))

  def post_update_params(self):
    self.iterations += 1
